# AccessibleDeepAgent: Neuroadaptive Accessibility Demo

**A hands-on walkthrough of the ADK (Accessibility Development Kit)**

This notebook demonstrates the **neuroadaptive accessibility pipeline** implemented in this repo. It focuses on the real ADK components in `src/adk`, including:

- **Signal normalization + cognitive state estimation** (Loops A & B)
- **Accessibility policy loop + UI adaptation** (Loop C)
- **Content refinement** (factuality, personalization, coherence)
- **Session metrics logging** (Loop E)

You can run this locally or in Colab. The demo uses the **heuristic pipeline** by default (no external services required).


## 0. Setup

This project has two dependency tiers:

- **Minimal** (for this notebook): `pydantic`, `numpy`, `aiohttp`, `pyyaml`, `nest_asyncio`
- **Full ADK stack**: `requirements-adk.txt` (includes PyTorch + optional memory backends)

The cell below installs the **minimal** dependencies and adds `src` to the Python path.


In [ ]:
import os
import sys
import subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB and not os.path.exists('AccessibleDeepAgent'):
    subprocess.check_call([
        'git', 'clone', 'https://github.com/Tuesdaythe13th/AccessibleDeepAgent.git'
    ])

if IN_COLAB:
    os.chdir('AccessibleDeepAgent')

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'pydantic', 'numpy', 'aiohttp', 'pyyaml', 'nest_asyncio'
])

if 'src' not in sys.path:
    sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

import nest_asyncio
nest_asyncio.apply()

print('✅ Setup complete')
print('📁 Working directory:', os.getcwd())


## 1. Import ADK Modules

We’ll use the **AccessibilityCoordinator** (top-level orchestrator), plus core schemas for signals and profiles.


In [ ]:
from adk.agents.core import AccessibilityCoordinator
from adk.utils import AccessibilityProfile, SignalType
from adk.tools.memory.memory_manager import MemoryManager

print('✅ Imports successful')


## 2. Create and Store an Accessibility Profile

The ADK uses **profiles** to tailor adaptations. Below we create a profile and store it in the in-memory CMS.


In [ ]:
import asyncio
from datetime import datetime

memory_manager = MemoryManager()

profile = AccessibilityProfile(
    profile_id='profile_demo_001',
    profile_name='Low-vision + focus support',
    user_id='user_demo',
    settings={
        'text_size': 'large',
        'contrast': 'high',
        'color_scheme': 'dark',
        'layout_density': 'comfortable'
    },
    cognitive_preferences={
        'summaries': True,
        'step_by_step': True
    },
    sensory_preferences={
        'reduce_motion': True,
        'audio_cues': False
    },
    interaction_preferences={
        'keyboard_first': True
    }
)

await memory_manager.save_accessibility_profile(profile)
saved = await memory_manager.get_accessibility_profile('user_demo')

print('✅ Profile saved')
print(saved)


## 3. Run a Single Accessibility Interaction

We now simulate a user interaction by sending signals (eye tracking, mouse movement, interaction timing). The coordinator will:

1. Normalize signals
2. Estimate cognitive state
3. Generate UI adaptations
4. Refine content for accessibility

This uses the built-in **heuristic estimator**—no external services required.


In [ ]:
coordinator = AccessibilityCoordinator()
await coordinator.initialize()

content_to_refine = (
    'The system might possibly be ready soon, so you should maybe check again later. ' 
    'This paragraph is dense and could use shorter sentences.'
)

signals = [
    (SignalType.EYE_TRACKING, 0.7, {'source': 'webcam'}),
    (SignalType.MOUSE_MOVEMENT, 0.8, {'variance': 0.6}),
    (SignalType.INTERACTION_TIMING, 0.65, {'avg_delay_ms': 900}),
    (SignalType.KEYBOARD_PATTERNS, 0.4, {'error_rate': 0.1}),
]

result = await coordinator.process_user_interaction(
    raw_signals=signals,
    user_id='user_demo',
    content_to_refine=content_to_refine,
    context={'page': 'settings', 'task': 'update preferences'}
)

result


### 3.1 Cognitive State Snapshot


In [ ]:
from pprint import pprint

print('Cognitive state:')
pprint(result['cognitive_state'])


### 3.2 UI Adaptations Generated


In [ ]:
print('UI adaptations:')
pprint(result['ui_adaptations'])


### 3.3 Refined Content


In [ ]:
refinement = result['content_refinement']
print('Original:')
print(refinement['original_content'])

print('
Refined:')
print(refinement['refined_content'])

print('
Changes:')
for change in refinement['all_changes'][:5]:
    print('-', change)


## 4. Multi-Interaction Simulation

To illustrate adaptation trends, we simulate multiple interactions with different signal patterns.


In [ ]:
async def simulate_interactions(coordinator):
    interaction_sets = [
        [
            (SignalType.INTERACTION_TIMING, 0.3, {'avg_delay_ms': 300}),
            (SignalType.EYE_TRACKING, 0.2, {'fixation': 0.9}),
        ],
        [
            (SignalType.INTERACTION_TIMING, 0.7, {'avg_delay_ms': 1100}),
            (SignalType.MOUSE_MOVEMENT, 0.9, {'variance': 0.8}),
        ],
        [
            (SignalType.INTERACTION_TIMING, 0.5, {'avg_delay_ms': 700}),
            (SignalType.KEYBOARD_PATTERNS, 0.6, {'error_rate': 0.2}),
        ],
    ]

    snapshots = []
    for idx, signals in enumerate(interaction_sets, start=1):
        result = await coordinator.process_user_interaction(
            raw_signals=signals,
            user_id='user_demo',
            content_to_refine=None
        )
        snapshots.append({
            'interaction': idx,
            'cognitive_state': result['cognitive_state'],
            'accessibility_score': result['metrics']['accessibility_score']
        })
    return snapshots

snapshots = await simulate_interactions(coordinator)
pprint(snapshots)


---

## Next Steps

- Explore the **full ADK demo**: `python src/adk/run_accessibility_agent.py --mode demo --user-id user123`
- Review advanced bias-mitigation examples in `src/adk/examples/`
- Read the ADK docs in `src/adk/docs/README.md`

If you need the full stack (torch + fairness metrics), install:

```bash
pip install -r requirements-adk.txt
```
